# One report through Gentrex

Gentrex turns the PDF of a genetic test report into one record. This
notebook follows one benchmark report through the four commands that do it.
For each command it shows what the command reads and what it writes.
[analysis.ipynb](analysis.ipynb) then shows how well the commands did on all
reports.

| Command | What it does | Model |
| --- | --- | --- |
| `gentrex transcribe` | Reads the text of every page. Scanned pages go through OCR. | LightOnOCR-3-4B, for scanned pages |
| `gentrex select` | Keeps the pages with the result, the patient's clinical features or the dates of the report. | Gemma 4 26B-A4B |
| `gentrex extract` | Turns the kept pages into one record, with the values as printed. | Gemma 4 31B |
| `gentrex normalize` | Adds standard values: ISO dates, categories and IDs. | SapBERT and EmbeddingGemma 2, on the CPU |

![The four commands of Gentrex, from PDF pages to a record with standard values](../docs/images/gentrex-arch.svg)

The notebook reads the run folders under `results/`. Make a run first with
`evaluation/local/run_pipeline.sh` or the Minerva job
([`evaluation/`](../evaluation/README.md)), and open the notebook from the
Gentrex folder:

```bash
uv run --group notebook jupyter lab evaluation_results/walkthrough.ipynb
```

The tables quote values from the report. After a run on real reports, clear
the outputs before you save or share the notebook.

In [ ]:
# The notebook group provides pandas and matplotlib.
# pyright: reportMissingImports=false

import json
import pathlib

import benchruns as bench
import pandas as pd
import pypdfium2

runs = bench.load_runs()
pd.DataFrame(
    [
        {"Setup": bench.setup_of(run), "Split": run["split"], "Reports": run["reports"]}
        for run in runs.values()
    ],
    index=list(runs),
)

## Choose a run and a report

The table above lists the scored runs, each named by its folder under
`results/gentrex-bench/`. `Setup` is the copy of the reports that the run
read, such as `native` or `scan`.

`RUN` is the run to follow and `REPORT` one of its reports. By default these
are the newest run of the native copy and its first report. To follow
another report, put a folder name from the table and a report name such as
`"SYN-0004"` in their place, and run the notebook again. A run of the scan
copy shows what OCR wrote.

In [ ]:
native = [name for name, run in runs.items() if bench.setup_of(run) == "native"]
RUN = (native or list(runs))[-1]
scores = runs[RUN]
REPORT = sorted(scores["by_report"])[0]


def of_report(command, name):
    """Returns the lines about REPORT in a file of a command's run folder."""
    if command not in scores["runs"]:
        print(f"{RUN} skipped gentrex {command}")
    found = bench.records(scores, command, name)
    if command in scores["runs"] and not found:
        print(f"{name} of gentrex {command} is not on this machine")
    return [record for record in found if record["file"] == f"{REPORT}.pdf"]


def first_existing(*paths):
    """Returns the first of the paths that is on this machine, or None."""
    return next((path for path in paths if path.exists()), None)


# The benchmark's folder here, for a run that was scored on another machine.
VERSION = bench.ROOT / "data" / "gentrex-bench" / scores["benchmark"]["version"]
print(
    f"{RUN}: {scores['reports']} reports of the split {scores['split']},"
    f" {bench.setup_of(scores)}; following {REPORT}"
)

## The report

A benchmark report is synthetic. Code picked every value that it prints, and
a language model wrote only the text around them, so its true content is
known. The report's truth file names the template that it was printed with
(`lab`), its kind of test, its result (`scenario`) and its challenge tags.
Each tag makes the report harder in one known way.

In [ ]:
truth_path = first_existing(
    pathlib.Path(scores["truth"]) / f"{REPORT}.json",
    VERSION / "truth" / f"{REPORT}.json",
)
case = json.loads(truth_path.read_text(encoding="utf-8"))["case"] if truth_path else {}
pd.Series(case, name=REPORT, dtype=object).drop("writer", errors="ignore").to_frame()

The first page of the PDF that `gentrex transcribe` read:

In [ ]:
pdf_path = first_existing(
    pathlib.Path(scores["pdfs"]) / f"{REPORT}.pdf",
    VERSION / "pdf" / bench.copy_of(scores) / f"{REPORT}.pdf",
)
if pdf_path is None:
    print(f"{REPORT}.pdf is not on this machine")
pypdfium2.PdfDocument(pdf_path)[0].render(scale=1).to_pil() if pdf_path else None

## `gentrex transcribe`

**Reads** the PDF. **Writes** the text of every page, one line of
`pages.jsonl` per page.

A page with a text layer is `native`, and its text is read from the PDF. A
page without one is `scanned`. It is drawn as an image and sent to the OCR
model, which writes its text. The native copy of the benchmark has only
native pages, so a run on it sends nothing to OCR. The scan, fax and image
copies have only scanned pages.

If text is lost here, no later command can get it back.

In [ ]:
pages = of_report("transcribe", "pages.jsonl")
pd.DataFrame(
    [
        {
            "page": page["page"],
            "source": page["source"],
            "characters": len(page["text"]),
            "error": page["error"],
        }
        for page in pages
    ]
)

In [ ]:
# The start of the first page's text.
print(pages[0]["text"][:700] if pages else "")

## `gentrex select`

**Reads** the text of every page. **Writes** a decision for every page, and
a copy of the pages that it keeps.

Most pages of a report only describe the test, its methods and its limits.
`gentrex select` asks the model yes-or-no questions about every page:

- `has_result` asks whether the page states the result or lists variants.
- `has_phenotype` asks whether it describes the patient's own clinical
  features.
- `has_dates` asks whether it lists the dates of the specimen and the
  report.

Each number in the table is the probability of "yes", read from the model's
first token. A page is kept when one of them is at least 0.5
(`select.keep_threshold`).

Fewer pages make the next request shorter and leave out text that could
mislead the model, such as a variant from a cited study. A page that is
wrongly dropped takes its values with it.

In [ ]:
pd.DataFrame(
    [
        {"page": d["page"], **d["p_yes"], "keep": d["keep"], "error": d["error"]}
        for d in of_report("select", "decisions.jsonl")
    ]
)

## `gentrex extract`

**Reads** the kept pages. **Writes** one record for the report.

The kept pages go to the model in one request, together with the extraction
schema, which describes every field. The model server lets the model write
only JSON that follows the schema. A field that the report doesn't state
comes back empty.

Fields ending in `_raw` hold the report's own wording. The field after each
holds the closest of a fixed list of values, as `classification_raw` and
`classification` do. The first table shows the fields of the report itself.

In [ ]:
extraction = (of_report("extract", "extractions.jsonl") or [{}])[0]
data = extraction.get("data") or {}
pd.Series(
    {k: v for k, v in data.items() if k not in ("variants", "patient_phenotypes")},
    name="value",
    dtype=object,
).to_frame()

The variants of the report, one column each:

In [ ]:
pd.DataFrame(data.get("variants") or []).T

The patient's clinical features. Before them comes `not_in_pages`, the
copied values that Gentrex couldn't find in the text of the pages. A value
in this list was made up or reworded by the model, so it is the first to
check against the PDF.

In [ ]:
print("not_in_pages:", extraction.get("not_in_pages"))
pd.DataFrame(data.get("patient_phenotypes") or [])

## `gentrex normalize`

**Reads** the record. **Writes** the record with standard values next to
the printed ones.

Laboratories print the same thing in many ways. `gentrex normalize` adds,
without a model server:

- dates as YYYY-MM-DD, a category for the specimen and one for the result,
- the gene's approved HGNC symbol and ID, and the disease's MONDO ID,
- whether each c. and p. change follows the HGVS syntax, and the variant's
  ClinVar ID,
- the closest HPO term for each clinical feature.

The first table puts the printed values next to the standard ones.

In [ ]:
record = (of_report("normalize", "normalized.jsonl") or [{}])[0]
normalized = record.get("normalized") or {}
pd.DataFrame(
    [
        {
            "field": field,
            "printed": data.get("overall_result_raw" if "result" in field else field),
            "normalized": normalized.get(field),
        }
        for field in (
            "date_of_birth",
            "collection_date",
            "received_date",
            "report_date",
            "specimen_type",
            "overall_result",
        )
    ]
).set_index("field")

What it adds to each variant:

In [ ]:
pd.DataFrame(normalized.get("variants") or []).T

The HPO term of each clinical feature, with whether the two embedding
models agreed on it. The table is empty after a run with `--no-hpo`.

In [ ]:
pd.DataFrame(
    [
        {
            "feature": feature["text"],
            "hpo_id": term["hpo_id"],
            "label": term["label"],
            "models_agree": term["models_agree"],
        }
        for feature, term in zip(
            data.get("patient_phenotypes") or [],
            normalized.get("patient_phenotypes") or [],
            strict=False,
        )
    ]
)

## The score of this report

The scorer compares the record with the truth file, value by value. A value
is correct, missed, wrong or extra. Each value that isn't correct is charged
to the first command that lost it:

- `gentrex transcribe`, if its text is not in the transcribed pages,
- `gentrex select`, if it is on a page that wasn't kept,
- `gentrex extract`, if the record has it wrong,
- `gentrex normalize`, if the record is right and the standard value is not.

The report counts as correct when nothing clinically relevant is wrong.
That covers the result category, the karyotype, and every variant with its
gene, DNA change, copy number, classification and zygosity.

The table lists the values of this report that aren't correct. `flagged`
says whether `gentrex extract` named the value in `not_in_pages`.

In [ ]:
summary = scores["by_report"][REPORT]
print(
    f"{REPORT}: {'correct' if summary['correct'] else 'not correct'};"
    f" variants matched {summary['variants']['matched']},"
    f" missed {summary['variants']['missed']}, extra {summary['variants']['extra']}"
)
errors = bench.errors(RUN)
errors[errors["report"] == REPORT].drop(columns="report")

[analysis.ipynb](analysis.ipynb) counts these outcomes over all reports of a
run, and compares the runs.